# Efficient GRPO Training with TRL + vLLM (Server Mode)

This notebook implements the Hugging Face cookbook pipeline for **GRPO online training with vLLM**.

## Goal
- Run **vLLM generation** on **GPU0** (fast sampling)
- Run **GRPO training** on **GPUs1–3** (policy updates)

## Why this is fast
GRPO spends most of its time generating completions. vLLM accelerates generation dramatically.

## Assumptions
- You are on a machine with **4 GPUs**
- You can run background processes from the notebook
- You have Hugging Face access (for Qwen) and optionally W&B


## 0) Install dependencies

If your environment already has these, you can skip.

In [ ]:
# !pip -q install -U "trl[vllm]" peft math_verify datasets accelerate transformers wandb

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xformers 0.0.32.post1 requires torch==2.8.0, but you have torch 2.9.0 which is incompatible.


## 1) Check GPUs

Confirm GPU visibility.

In [1]:
!nvidia-smi

NVIDIA-SMI has failed because it couldn't communicate with the NVIDIA driver. Make sure that the latest NVIDIA driver is installed and running.



## 2) (Optional) W&B login

If you're already logged in (via `.netrc`), you can skip.

In [2]:
# !wandb login
import wandb
print("wandb version:", wandb.__version__)

wandb version: 0.25.0


## 3) Start vLLM server on GPU0 (background)

This cell launches the vLLM server in the background and writes logs to `vllm_server.log`.

If you need to restart it, run the stop cell first (below) and then rerun this cell.

In [11]:
import os, subprocess, textwrap, time, signal

MODEL_ID = "Qwen/Qwen2-0.5B-Instruct"
PID_FILE = "vllm_server.pid"
LOG_FILE = "vllm_server.log"

# If we previously started a server, stop only that one
if os.path.exists(PID_FILE):
    pid = int(open(PID_FILE).read().strip())
    try:
        os.kill(pid, signal.SIGTERM)
        print(f"Stopped previous vLLM server pid={pid}")
    except ProcessLookupError:
        print(f"PID file existed but process {pid} not running anymore")
    os.remove(PID_FILE)

cmd = f"""
CUDA_VISIBLE_DEVICES=0 nohup trl vllm-serve \
  --model {MODEL_ID} \
  --host 127.0.0.1 \
  --port 8000 \
  --gpu-memory-utilization 0.2 \
  --max-model-len 2048 \
  > {LOG_FILE} 2>&1 &
echo $! > {PID_FILE}
"""

print("Starting vLLM server...\n", textwrap.dedent(cmd))
subprocess.run(cmd, shell=True, check=True)

time.sleep(2)
pid = open(PID_FILE).read().strip()
print(f"vLLM server started pid={pid}. Logs: {LOG_FILE}")
subprocess.run(f"tail -n 30 {LOG_FILE}", shell=True)


Stopped previous vLLM server pid=876853
Starting vLLM server...
 
CUDA_VISIBLE_DEVICES=0 nohup trl vllm-serve   --model Qwen/Qwen2-0.5B-Instruct   --host 127.0.0.1   --port 8000   --gpu-memory-utilization 0.2   --max-model-len 2048   > vllm_server.log 2>&1 &
echo $! > vllm_server.pid

vLLM server started pid=878569. Logs: vllm_server.log


CompletedProcess(args='tail -n 30 vllm_server.log', returncode=0)

### 3.1 Verify vLLM server is responding

We check the `/v1/models` endpoint.

In [15]:
!curl -s http://127.0.0.1:8000/v1/models | head -c 1000
!ss -ltnp | grep ':8000' || echo "Nothing listening on 8000"


Nothing listening on 8000


In [18]:
!tail -n 80 vllm_server.log


[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.16s/it]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.16s/it]
(EngineCore_DP0 pid=878856) 
(EngineCore_DP0 pid=878856) ERROR 02-17 12:45:13 [core.py:708] EngineCore failed to start.
(EngineCore_DP0 pid=878856) ERROR 02-17 12:45:13 [core.py:708] Traceback (most recent call last):
(EngineCore_DP0 pid=878856) ERROR 02-17 12:45:13 [core.py:708]   File "/home/yuen_chen/anaconda3/envs/enco/lib/python3.12/site-packages/vllm/v1/engine/core.py", line 699, in run_engine_core
(EngineCore_DP0 pid=878856) ERROR 02-17 12:45:13 [core.py:708]     engine_core = EngineCoreProc(*args, **kwargs)
(EngineCore_DP0 pid=878856) ERROR 02-17 12:45:13 [

### 3.2 Stop vLLM server (if needed)

## 4) Write the GRPO training script

We train on GPUs 1–3 while vLLM serves generation on GPU0.

The script:
- Loads a small slice of `AI-MO/NuminaMath-TIR`
- Builds a **string** `prompt` column (TRL 0.28.0 expects `prompt`)
- Adds LoRA
- Uses **format + accuracy** rewards
- Enables TRL vLLM integration in **server** mode

Includes a workaround for the TRL profiling module referencing `wandb` without importing it.

In [ ]:
%%writefile train_grpo_vllm_server.py
import os
import re
import torch
import wandb

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from math_verify import LatexExtractionConfig, parse, verify
from trl import GRPOConfig, GRPOTrainer

# --- Workaround for TRL profiling NameError: wandb referenced without import in trl.extras.profiling ---
import trl.extras.profiling as trl_profiling
trl_profiling.wandb = wandb

dataset_id = "AI-MO/NuminaMath-TIR"
train_dataset, test_dataset = load_dataset(dataset_id, split=["train[:5%]", "test[:5%]"])

SYSTEM_PROMPT = (
    "A conversation between User and Assistant. The user asks a question, and the Assistant solves it. "
    "The assistant first thinks about the reasoning process in the mind and then provides the user with the answer. "
    "The reasoning process and answer are enclosed within <think> and </think> tags, and <answer> and </answer> tags, respectively." 
)

model_id = "Qwen/Qwen2-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def add_prompt(example, max_prompt_tokens=128):
    text = (
        "system\n" + SYSTEM_PROMPT + "\n"
        "user\n" + example["problem"] + "\n"
        "assistant\n"
    )
    ids = tokenizer(text, truncation=True, max_length=max_prompt_tokens)["input_ids"]
    return {"prompt": tokenizer.decode(ids, skip_special_tokens=True)}

train_dataset = train_dataset.map(add_prompt, fn_kwargs={"max_prompt_tokens": 128})
test_dataset  = test_dataset.map(add_prompt, fn_kwargs={"max_prompt_tokens": 128})
train_dataset = train_dataset.remove_columns([c for c in train_dataset.column_names if c not in ["prompt", "solution"]])
test_dataset  = test_dataset.remove_columns([c for c in test_dataset.column_names if c not in ["prompt", "solution"]])

model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype="auto", device_map="auto")

lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"],
)
model = get_peft_model(model, lora_config)

FORMAT_RE = re.compile(r"(?s)^\s*<think>.*?</think>\s*<answer>.*?</answer>\s*$")

def format_reward(completions, **kwargs):
    texts = [c[0]["content"] for c in completions]
    return [1.0 if FORMAT_RE.match(t) else 0.0 for t in texts]

def accuracy_reward(completions, **kwargs):
    sols = kwargs["solution"]
    texts = [c[0]["content"] for c in completions]
    out = []
    for text, sol in zip(texts, sols):
        gold = parse(sol, extraction_mode="first_match", extraction_config=[LatexExtractionConfig()])
        pred = parse(text, extraction_mode="first_match", extraction_config=[LatexExtractionConfig()])
        if len(gold) != 0:
            try:
                out.append(float(verify(pred, gold)))
            except Exception:
                out.append(0.0)
        else:
            out.append(1.0)
    return out

training_args = GRPOConfig(
    output_dir="Qwen2-0.5B-GRPO-vLLM-server",
    learning_rate=1e-5,
    remove_unused_columns=False,
    num_train_epochs=1,

    # IMPORTANT: set explicitly (default is 8)
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,

    bf16=torch.cuda.is_available(),

    max_completion_length=64,
    num_generations=4,

    report_to=["wandb"],
    run_name="qwen2-grpo-vllm-server",
    logging_steps=10,
    save_strategy="steps",
    save_steps=50,
    push_to_hub=False,

    # vLLM integration (server mode)
    use_vllm=True,
    vllm_mode="server",
    vllm_server_base_url="http://127.0.0.1:8000",
    vllm_server_timeout=240.0,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[format_reward, accuracy_reward],
    args=training_args,
    train_dataset=train_dataset,
)

trainer.train()
trainer.save_model(training_args.output_dir)
print("Saved to:", training_args.output_dir)


## 5) Launch training on GPUs 1–3

This uses `accelerate` to launch **3 processes** on GPUs 1,2,3.

Make sure the vLLM server is still running (Step 3).

In [ ]:
!CUDA_VISIBLE_DEVICES=1,2,3 accelerate launch --num_processes 3 train_grpo_vllm_server.py

## 6) Monitor progress

### W&B
- Go to https://wandb.ai
- Workspace/entity: the one shown in your login (e.g., `firstry`)
- Project: by default TRL uses `huggingface` unless overridden; our script will still show the run under your W&B settings.
- Run name: `qwen2-grpo-vllm-server`

### Local
- Watch GPUs:
  - `watch -n 1 nvidia-smi`
- Check vLLM logs:
  - `tail -f vllm_server.log`
